# Assigment 2 - Python dictionaries and file I/O

### Overview
------------

Most FASTA files you will encounter will contain multiple sequences. Further, for genes you will often get 3' and 5' untranslated sequences (UTRs) flanking the coding sequence. In this exercise you will be writing a function for parsing a multiple-sequence FASTA file and translating each sequence, locating the coding sequence in between the UTRs. Finally, you will look at the prevalence of amino acids across the different sequences.

Biological Learning Objectives

- Use a codon table and DNA sequences to identify and translate coding sequences
- Count and compare amino acid usage for the population of sequences 

Computational Learning Objectives

- Create and modify a dicitionary
- Use keys to retrieve values stored in a dictionary
- Step through each key and value in a dictionary inside a `for` loop
- Use conditionals to alter the behaviour or a `for` or `while` loop, terminating early if necessary
- Open and read a text file
- Write to a text file

### Instructions
----------------

- Save a copy of this notebook as `~/qbXX-answers/day2-morning/python_dictionaries.ipynb`.
- Copy the files [`sequences.fa`](https://raw.githubusercontent.com/bxlab/cmdb-quantbio/refs/heads/main/assignments/bootcamp/dictionaries_file_io/sequences.fa) and [`codons.tsv`](https://raw.githubusercontent.com/bxlab/cmdb-quantbio/refs/heads/main/assignments/bootcamp/dictionaries_file_io/codons.tsv) into `~/qbXX-answers/day2-morning/` (but do not submit them with your answer)
- Fill in answers in the available code/markdown cells below.
- Remember to comment your code to help yourself and us know what each part is intended to do.

### What to turn in
-------------------

- This filled-in notebook
- Your codon usage file

### Scoring
-----------

- 2.0 pts - Multiple-sequence fasta load function
- 0.5 pts - Codon table load function
- 4.0 pts - Translation function
    - 1.5 pts - Loop stepping through sequence with correct reading frame
    - 1.0 pts - Skipping 5' UTR
    - 0.5 pts - Converting coding sequence to amino acids
    - 1.0 pts - Skipping 3' UTR
- 2.0 pts - Counting amino acid usage
- 0.5 pts - Converting counts to percentages
- 1.0 pts - Writing usage results file

10 pts total

-------------------

1. Start by setting the correct working directory (this is important if you don't want to use full paths for your file names)


In [1]:
%cd ~/Desktop/quant_bootcamp/day2assignment/

/Users/cmdb/Desktop/quant_bootcamp/day2assignment


/opt/anaconda3/envs/qb26/lib/python3.12/site-packages/IPython/core/magics/osm.py:393: UserWarning: This is now an optional IPython functionality, using bookmarks requires you to install the `pickleshare` library.
  bkms = self.shell.db.get('bookmarks', {})
/opt/anaconda3/envs/qb26/lib/python3.12/site-packages/IPython/core/magics/osm.py:417: UserWarning: This is now an optional IPython functionality, setting dhist requires you to install the `pickleshare` library.
  self.shell.db['dhist'] = compress_dhist(dhist)[-100:]


2. Building on the [code](https://github.com/bxlab/cmdb-quantbio/blob/main/lectures/python_dicts_file_io/livecoding.ipynb) for reading in a single FASTA sequence, adapt it to read in multiple sequences from a single FASTA file, storing each sequence in a dictionary using the sequence name as the key. Wrap this code in a function such that it takes a file name as the only function argument and returns the dictionary of sequences.

- To check if a line represents the start of a new sequence, consider using the string method `.startswith()`
- Don't forget to close your filestream

In [2]:
def read_fa(filename):
    fs = open(filename)

    sequence = []
    name = None
    fasta = {}

    for line in fs:
        # Remove whitespace and newline characters
        line = line.rstrip()

        if line.startswith(">"):
            # Save the previous sequence before starting a new one
            if name is not None:
                fasta[name] = "".join(sequence)
            # Record the new sequence name
            name = line.lstrip("> ")
            # Reset the sequence list for the new sequence
            sequence = []
        else:
            # Add the current line to the sequence
            sequence.append(line)

    # Save the final sequence in the file
    if name is not None:
        fasta[name] = "".join(sequence)

    fs.close()
    return fasta

# Test the function
read_fa("sequences.fa")


{'CCDS2.2|Hs110|chr1': 'ATGTCCAAGGGGATCCTGCAGGTGCATCCTCCGATCTGCGACTGCCCGGGCTGCCGAATATCCTCCCCGGTGAACCGGGGGCGGCTGGCAGACAAGAGGACAGTCGCCCTGCCTGCCGCCCGGAACCTGAAGAAGGAGCGAACTCCCAGCTTCTCTGCCAGCGATGGTGACAGCGACGGGAGTGGCCCCACCTGTGGGCGGCGGCCAGGCTTGAAGCAGGAGGATGGTCCGCACATCCGTATCATGAAGAGAAGAGTCCACACCCACTGGGACGTGAACATCTCTTTCCGAGAGGCGTCCTGCAGCCAGGACGGCAACCTTCCCACCCTCATATCCAGCGTCCACCGCAGCCGCCACCTCGTTATGCCCGAGCATCAGAGCCGCTGTGAATTCCAGAGAGGCAGCCTGGAGATTGGCCTGCGACCCGCCGGTGACCTGTTGGGCAAGAGGCTGGGCCGCTCCCCCCGTATCAGCAGCGACTGCTTTTCAGAGAAGAGGGCACGAAGCGAATCGCCTCAAGAGGCGCTGCTGCTGCCGCGGGAGCTGGGGCCCAGCATGGCCCCGGAGGACCATTACCGCCGGCTTGTGTCAGCACTGAGCGAGGCCAGCACCTTTGAGGACCCTCAGCGCCTCTACCACCTGGGCCTCCCCAGCCACGGTGAGGACCCACCCTGGCATGATCCCCCTCATCACCTCCCCAGCCACGATCTCCTGAGGGTCCGGCAGGAGGTGGCGGCTGCAGCTCTGAGGGGCCCCAGTGGCCTGGAAGCCCACCTGCCCTCCTCCACGGCAGGTCAGCGTCGGAAGCAGGGCCTGGCTCAGCACCGGGAGGGCGCCGCCCCAGCTGCCGCCCCGTCCTTCTCGGAGAGGGAGCTGCCTCAGCCGCCCCCCTTGCTGTCGCCGCAGAATGCCCCTCACGTCGCCCTGGGCCCCCATCTCAGGCCCCCCTTCCTGGGGGTGCCCTCGGCTCTGTGCC

3. Wrap the code for reading in the codon table into a function, taking a file name in as the argument and returning the dictionary of codon/amin acid pairs.

In [3]:
def read_codon(filename):
    codon_fname = open(filename)
    codons = {}
    for line in codon_fname:
        codon, aa = line.rstrip().split('\t')
        codons[codon] = aa

    codon_fname.close()
    return codons

# Test the function
codons = read_codon('codons.tsv')
print(codons)


{'TTT': 'F', 'TTC': 'F', 'TTA': 'L', 'TTG': 'L', 'TAT': 'Y', 'TAC': 'Y', 'TAA': '*', 'TAG': '*', 'CTT': 'L', 'CTC': 'L', 'CTA': 'L', 'CTG': 'L', 'CAT': 'H', 'CAC': 'H', 'CAA': 'Q', 'CAG': 'Q', 'ATT': 'I', 'ATC': 'I', 'ATA': 'I', 'ATG': 'M', 'AAT': 'N', 'AAC': 'N', 'AAA': 'K', 'AAG': 'K', 'GTT': 'V', 'GTC': 'V', 'GTA': 'V', 'GTG': 'V', 'GAT': 'D', 'GAC': 'D', 'GAA': 'E', 'GAG': 'E', 'TCT': 'S', 'TCC': 'S', 'TCA': 'S', 'TCG': 'S', 'TGT': 'C', 'TGC': 'C', 'TGA': '*', 'TGG': 'W', 'CCT': 'P', 'CCC': 'P', 'CCA': 'P', 'CCG': 'P', 'CGT': 'R', 'CGC': 'R', 'CGA': 'R', 'CGG': 'R', 'ACT': 'T', 'ACC': 'T', 'ACA': 'T', 'ACG': 'T', 'AGT': 'S', 'AGC': 'S', 'AGA': 'R', 'AGG': 'R', 'GCT': 'A', 'GCC': 'A', 'GCA': 'A', 'GCG': 'A', 'GGT': 'G', 'GGC': 'G', 'GGA': 'G', 'GGG': 'G'}


4. Write a function for translating the CDS sequences into amino acid sequences. This function will need to take in two arguments, the codon table and the DNA sequence. The DNA sequences contain untranslated sequences (UTRs) at the start and end so you will need to step through the sequences to find the first methionine (M). Likewise, you will need to stop when you encounter the first step codon (*) rather than translating through the end of the sequence.

- Using a `while` loop may be useful for this task, but it is not required as `for` loops can also work
- You will need to consider three different parts reading the sequence:
    1. Have you reached the start of the coding sequence
    2. Do you need to record the current codon's amino acid
    3. Have you reached the end of the coding sequence

In [ ]:
def translateCDS(codon_table, DNA_sequence):
    AAs = []
    start_pos = None

    # Find the first start codon in the correct reading frame
    for i in range(0, len(DNA_sequence) - 2, 3):
        codon = DNA_sequence[i:i + 3]
        if codon == "ATG":
            start_pos = i
            break

    # Return an empty string if there is no start codon
    if start_pos is None:
        return ""

    # Translate from the start codon
    for i in range(start_pos, len(DNA_sequence) - 2, 3):
        codon = DNA_sequence[i:i + 3]
        aa = codon_table[codon]
        # Stop at the first stop codon
        if aa == "*":
            break

        AAs.append(aa)

    return "".join(AAs)


### This is for test
test_DNA = "AAAATGTTTGCTTAACCC"

test_protein = translateCDS(codons, test_DNA)

print(test_protein)

test_DNA2 = "AAATTTGCTTAACCC"

test_protein2 = translateCDS(codons, test_DNA2)

print(test_protein2)


MFA



5. Finally, put it all together, loading in the FASTA sequences and codon table, and translating the into amino acids. Once you have the amino acid sequences, count the number of times each amino acid is used. Finally, convert these counts into percentages and write them to a tab-separated file with the first column being the amino acid letter and the second column being the percent usage.

- To get the percentages, it will helpful to keep a running total of the number of amino acids as you find the counts
- The dictionary method `.setdefault` may be useful for intializing you count dictionary for each new amino acid
- Don't forget to close your filestream

In [5]:
# Load in the FASTA sequences and codon table
DNA_seq = read_fa("sequences.fa")
codons = read_codon('codons.tsv')

# Translate all DNA sequences
protein_seq = {}

for name, sequence in DNA_seq.items():
    protein = translateCDS(codons, sequence)
    protein_seq[name] = protein

# Count the num of each amino acid is used
counter = {}
for name, protein in protein_seq.items():
    for aa in protein:
        counter.setdefault(aa, 0)
        counter[aa] += 1
print(counter)

# Convert into percentages
total = 0
protein_percentage = {}
# Calculate the total num of amino acids
for number in counter.values():
    total += number
print(total)
# Percentage Dict
for aa, number in counter.items():
    protein_percentage.setdefault(aa, 0)
    protein_percentage[aa] = number / total

print(protein_percentage)


{'M': 10893, 'S': 40785, 'K': 28226, 'G': 34500, 'I': 21519, 'L': 50266, 'Q': 24274, 'V': 30105, 'H': 12731, 'P': 32677, 'C': 11762, 'D': 24077, 'R': 28415, 'N': 17677, 'A': 35378, 'T': 26412, 'E': 35094, 'F': 18532, 'W': 6203, 'Y': 13880}
503406
{'M': 0.0216385978713007, 'S': 0.08101810467098128, 'K': 0.056070050813856014, 'G': 0.06853315216743543, 'I': 0.04274680873887081, 'L': 0.09985180947386404, 'Q': 0.048219528571371816, 'V': 0.05980262452175779, 'H': 0.025289726383873057, 'P': 0.06491182067754457, 'C': 0.023364838718648567, 'D': 0.04782819434015487, 'R': 0.05644549329964283, 'N': 0.035114797996050905, 'A': 0.0702772712283922, 'T': 0.052466597537574045, 'E': 0.06971311426562257, 'F': 0.03681322828889604, 'W': 0.012322062112887014, 'Y': 0.02757217832127547}
